In [1]:
###############################################
#   Código para Colab
###############################################

import os
from pathlib import Path
import cv2
import glob
import numpy as np
from tqdm import tqdm

# ============================
# Ф Features
# ============================

def extract_features_opencv(img):
    """Extrae descriptores básicos"""
    feats = []

    # Escala de grises
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Redimensionar
    img = cv2.resize(img, (256, 256))

    # 1. Histograma normalizado
    hist = cv2.calcHist([img], [0], None, [32], [0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    feats.extend(hist)

    # 2. Bordes Canny (promedio)
    edges = cv2.Canny(img, 100, 200)
    feats.append(np.mean(edges))

    # 3. Varianza como descriptor de textura
    feats.append(np.var(img))

    return np.array(feats, dtype=np.float32)


# =====================================
# Ф Procesar dataset en lotes
# =====================================

def extract_features_batches(base_dir, batch_size=300):
    all_features = []
    all_labels = []
    all_paths = []

    classes = {"NORMAL": 0, "PNEUMONIA": 1}

    # 1. Recolectar todas las rutas
    image_paths = []
    for split in ["train", "val", "test"]:
        split_path = os.path.join(base_dir, split)
        if not os.path.isdir(split_path):
            continue
        for cname, label in classes.items():
            class_dir = os.path.join(split_path, cname)
            if os.path.isdir(class_dir):
                for f in (glob.glob(os.path.join(class_dir, "*.jpeg")) +
                          glob.glob(os.path.join(class_dir, "*.jpg")) +
                          glob.glob(os.path.join(class_dir, "*.png"))):
                    image_paths.append((f, label))

    print("Total imágenes encontradas:", len(image_paths))

    # 2. Procesar por lotes
    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch = image_paths[i:i+batch_size]

        feats_batch = []
        labels_batch = []
        paths_batch = []

        for path, label in batch:
            try:
                img = cv2.imread(path)
                if img is None:
                    print("[ERROR] No se pudo leer:", path)
                    continue

                feats = extract_features_opencv(img)

                feats_batch.append(feats)
                labels_batch.append(label)
                paths_batch.append(path)

            except Exception as e:
                print("[ERROR procesando]", path, e)

        # Agregar lote procesado
        if len(feats_batch) > 0:
            all_features.append(np.vstack(feats_batch))
            all_labels.extend(labels_batch)
            all_paths.extend(paths_batch)

        # liberar RAM del batch
        del feats_batch, labels_batch, paths_batch

    # 3. Unir todo
    X = np.vstack(all_features)
    y = np.array(all_labels)

    return X, y, all_paths


#########################################
# EJECUTAR PIPELINE
#########################################

DATA_DIR = Path('../data/chest-xray-pneumonia/chest_xray')

X, y, paths = extract_features_batches(DATA_DIR, batch_size=300)

print("\n Features shape:", X.shape)
print("Labels shape:", y.shape)
print("Ejemplo de path:", paths[0])


Total imágenes encontradas: 5856


100%|██████████| 20/20 [00:25<00:00,  1.30s/it]


 Features shape: (5856, 34)
Labels shape: (5856,)
Ejemplo de path: ../data/chest-xray-pneumonia/chest_xray/train/NORMAL/NORMAL2-IM-0927-0001.jpeg


In [4]:
###############################################
#   TALLER — PARTE DE MODELADO
###############################################

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

BASE_DIR = Path('../results/')
# ======================
# NORMALIZACIÓN
# ======================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, os.path.join(BASE_DIR, "scaler.pkl"))
print("✔ Escalado OK:", X_scaled.shape)


# ======================
# PCA 95% VAR
# ======================
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)
joblib.dump(pca, os.path.join(BASE_DIR, "pca_95.pkl"))

print("✔ PCA OK:", X_pca.shape)


# ======================
# SMOTE
# ======================
print("Antes SMOTE:", np.bincount(y))
sm = SMOTE()
X_bal, y_bal = sm.fit_resample(X_pca, y)
print("Después SMOTE:", np.bincount(y_bal))


# ======================
# CLASIFICADORES
# ======================
models = {
    "SVM_RBF": SVC(kernel="rbf", probability=True),
    "SVM_LINEAR": SVC(kernel="linear", probability=True),
    "RandomForest": RandomForestClassifier(n_estimators=200),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "LogisticRegression": LogisticRegression(max_iter=2000)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}

for name, clf in models.items():
    print("\n===============================")
    print("Entrenando:", name)
    print("===============================")

    y_pred = cross_val_predict(clf, X_bal, y_bal, cv=skf)
    y_prob = cross_val_predict(clf, X_bal, y_bal, cv=skf, method="predict_proba")

    acc = accuracy_score(y_bal, y_pred)
    prec = precision_score(y_bal, y_pred)
    rec = recall_score(y_bal, y_pred)
    f1 = f1_score(y_bal, y_pred)
    cm = confusion_matrix(y_bal, y_pred)

    fpr, tpr, _ = roc_curve(y_bal, y_prob[:, 1])
    roc_auc = auc(fpr, tpr)

    results[name] = {
        "acc": acc,
        "prec": prec,
        "rec": rec,
        "f1": f1,
        "roc_auc": roc_auc,
        "confusion_matrix": cm.tolist()
    }

    print("\n>>> RESULTADOS", name)
    print(classification_report(y_bal, y_pred))
    print("AUC:", roc_auc)


# ======================
# GUARDAR RESULTADOS
# ======================
df_results = pd.DataFrame(results).T
df_results.to_csv(f"{BASE_DIR}/resultados_modelos.csv")
joblib.dump(results, f"{BASE_DIR}/model_results.pkl")

print("\n PROCESO FINALIZADO")
print("➡ Se guardaron:")
print(f" - {os.path.join(BASE_DIR, 'scaler.pkl')}")
print(f" - {os.path.join(BASE_DIR, 'pca_95.pkl')}")
print(f" - {os.path.join(BASE_DIR, 'resultados_modelos.csv')}")
print(f" - {os.path.join(BASE_DIR, 'model_results.pkl')}")


✔ Escalado OK: (5856, 34)
✔ PCA OK: (5856, 15)
Antes SMOTE: [1583 4273]
Después SMOTE: [4273 4273]

Entrenando: SVM_RBF


/Users/carlosviera/Documents/GitHub/clasificacion-imagenes-medicas/venv/lib/python3.11/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: divide by zero encountered in matmul
  C = X.T @ X
/Users/carlosviera/Documents/GitHub/clasificacion-imagenes-medicas/venv/lib/python3.11/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: overflow encountered in matmul
  C = X.T @ X
/Users/carlosviera/Documents/GitHub/clasificacion-imagenes-medicas/venv/lib/python3.11/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: invalid value encountered in matmul
  C = X.T @ X
/Users/carlosviera/Documents/GitHub/clasificacion-imagenes-medicas/venv/lib/python3.11/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/carlosviera/Documents/GitHub/clasificacion-imagenes-medicas/venv/lib/python3.11/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encou


>>> RESULTADOS SVM_RBF
              precision    recall  f1-score   support

           0       0.89      0.94      0.92      4273
           1       0.94      0.88      0.91      4273

    accuracy                           0.91      8546
   macro avg       0.91      0.91      0.91      8546
weighted avg       0.91      0.91      0.91      8546

AUC: 0.9603377687216752

Entrenando: SVM_LINEAR

>>> RESULTADOS SVM_LINEAR
              precision    recall  f1-score   support

           0       0.83      0.90      0.87      4273
           1       0.89      0.82      0.85      4273

    accuracy                           0.86      8546
   macro avg       0.86      0.86      0.86      8546
weighted avg       0.86      0.86      0.86      8546

AUC: 0.9155737025693581

Entrenando: RandomForest

>>> RESULTADOS RandomForest
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      4273
           1       0.95      0.90      0.93      4273

    

/Users/carlosviera/Documents/GitHub/clasificacion-imagenes-medicas/venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/carlosviera/Documents/GitHub/clasificacion-imagenes-medicas/venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/carlosviera/Documents/GitHub/clasificacion-imagenes-medicas/venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/carlosviera/Documents/GitHub/clasificacion-imagenes-medicas/venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/carlosviera/Docu

La construcción del pipeline permitió transformar un conjunto de imágenes médicas en una representación numérica robusta basada en 34 descriptores clásicos, los cuales fueron estandarizados y reducidos a 15 componentes principales mediante PCA, conservando la mayor parte de la variabilidad original. El dataset presentaba un desbalance notable entre las clases NORMAL y PNEUMONIA, por lo que la aplicación de SMOTE no solo equilibró los tamaños de cada grupo, sino que también contribuyó a mejorar la estabilidad y el desempeño de los modelos entrenados.

Los métodos lineales, como SVM con kernel lineal y la regresión logística, mostraron un rendimiento aceptable alrededor del 85%, confirmando que la frontera de decisión del problema no es estrictamente lineal. En contraste, los modelos no lineales capturaron con mayor eficacia la complejidad inherente a las imágenes médicas. SVM con kernel RBF logró una precisión del 91% y un AUC cercano a 0.96, mientras que KNN obtuvo métricas comparables, beneficiándose de la estructura de proximidad en el espacio de características.

El mejor desempeño global lo alcanzó Random Forest, con una exactitud del 93% y el AUC más alto del conjunto, alrededor de 0.97. Este resultado refleja su capacidad para manejar relaciones no lineales, interacciones entre variables y variabilidad dentro de cada clase. En conjunto, el proceso completo, desde la extracción de descriptores hasta la evaluación final, demostró que el modelo es capaz de discriminar de manera confiable entre imágenes de tórax sanas y con neumonía, evidenciando la solidez del enfoque basado en características y mostrando un entendimiento profundo del flujo de trabajo en visión artificial y aprendizaje automático.